In [22]:
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = "../raw_csv/type1_time_stress"
OUT_DIR  = "../results"
os.makedirs(OUT_DIR, exist_ok=True)

print("BASE_DIR exists:", os.path.exists(BASE_DIR))

BASE_DIR exists: True


In [23]:
files = glob.glob(os.path.join(BASE_DIR, "**", "*.csv"), recursive=True)
print("Number of CSV files:", len(files))
print(files[:5])

dfs = []
for f in files:
    df = pd.read_csv(f)
    df["SourceFile"] = os.path.basename(f)
    dfs.append(df)

raw_df = pd.concat(dfs, ignore_index=True)
raw_df.head()

Number of CSV files: 9
['../raw_csv/type1_time_stress\\MeO2PACz_Ag_15h_0p8V.csv', '../raw_csv/type1_time_stress\\MeO2PACz_Ag_15h_1p5V.csv', '../raw_csv/type1_time_stress\\MeO2PACz_Ag_1h_0p8V.csv', '../raw_csv/type1_time_stress\\MeO2PACz_Ag_stepstress_0p2V.csv', '../raw_csv/type1_time_stress\\MeO2PACz_Ag_stepstress_0p4V.csv']


,Voltage,Current,SourceFile
0,1.20,-29.977912,MeO2PACz_Ag_15h_0p8V.csv
1,1.18,-29.977220,MeO2PACz_Ag_15h_0p8V.csv
2,1.16,-25.316807,MeO2PACz_Ag_15h_0p8V.csv
3,1.14,-13.363572,MeO2PACz_Ag_15h_0p8V.csv
4,1.12,-4.027651,MeO2PACz_Ag_15h_0p8V.csv


In [24]:
raw_df.columns = [c.strip() for c in raw_df.columns]

# Normalize column names
if "Voltage (V)" in raw_df.columns:
    raw_df = raw_df.rename(columns={"Voltage (V)": "Voltage"})
if "Current (mAcm-2)" in raw_df.columns:
    raw_df = raw_df.rename(columns={"Current (mAcm-2)": "Current"})

raw_df["Voltage"] = pd.to_numeric(raw_df["Voltage"], errors="coerce")
raw_df["Current"] = pd.to_numeric(raw_df["Current"], errors="coerce")
raw_df = raw_df.dropna(subset=["Voltage","Current"])

In [25]:
def parse_metadata(fname):
    name = fname.replace(".csv","")
    parts = name.split("_")

    meta = {"HTL":None,"Electrode":None,"StressMode":None,"StressVoltage":None,"StressDuration_s":None}

    if len(parts) >= 2:
        meta["HTL"] = parts[0]
        meta["Electrode"] = parts[1]

    for p in parts:
        if "h" in p:
            meta["StressDuration_s"] = float(p.replace("h","")) * 3600
        if "V" in p:
            meta["StressVoltage"] = float(p.replace("p",".").replace("V",""))
        if "step" in p.lower():
            meta["StressMode"] = "stepstress"

    if meta["StressMode"] is None:
        meta["StressMode"] = "constant"

    return pd.Series(meta)

meta_df = raw_df["SourceFile"].apply(parse_metadata)
type1_df = pd.concat([raw_df, meta_df], axis=1)
type1_df.head()

,Voltage,Current,SourceFile,HTL,Electrode,StressMode,StressVoltage,StressDuration_s
0,1.20,-29.977912,MeO2PACz_Ag_15h_0p8V.csv,MeO2PACz,Ag,constant,0.8,54000.0
1,1.18,-29.977220,MeO2PACz_Ag_15h_0p8V.csv,MeO2PACz,Ag,constant,0.8,54000.0
2,1.16,-25.316807,MeO2PACz_Ag_15h_0p8V.csv,MeO2PACz,Ag,constant,0.8,54000.0
3,1.14,-13.363572,MeO2PACz_Ag_15h_0p8V.csv,MeO2PACz,Ag,constant,0.8,54000.0
4,1.12,-4.027651,MeO2PACz_Ag_15h_0p8V.csv,MeO2PACz,Ag,constant,0.8,54000.0


In [26]:
# --- FORCE canonical columns on type1_df (not raw_df) ---
type1_df.columns = [c.strip() for c in type1_df.columns]

rename_map = {
    "Voltage (V)": "V",
    "Voltage": "V",
    "Current (mAcm-2)": "J",
    "Current (mA cm-2)": "J",
    "Current": "J",
}
type1_df = type1_df.rename(columns={k: v for k, v in rename_map.items() if k in type1_df.columns})

# sanity check
print("Has V?", "V" in type1_df.columns, "| Has J?", "J" in type1_df.columns)
print("Columns:", type1_df.columns.tolist())

type1_df["V"] = pd.to_numeric(type1_df["V"], errors="coerce")
type1_df["J"] = pd.to_numeric(type1_df["J"], errors="coerce")
type1_df = type1_df.dropna(subset=["V", "J"])

Has V? True | Has J? True
Columns: ['V', 'J', 'SourceFile', 'HTL', 'Electrode', 'StressMode', 'StressVoltage', 'StressDuration_s']


In [27]:
I_THRESH = 50.0   # <-- increase threshold (you can tune)
VREF = 1.0

def extract_curve_features(g):
    g = g.sort_values("V")
    V = g["V"].to_numpy()
    J = g["J"].to_numpy()
    absJ = np.abs(J)

    idx = np.where(absJ >= I_THRESH)[0]
    Vbr = V[idx[0]] if len(idx) else np.nan

    Jref = J[np.argmin(np.abs(V - VREF))]

    return pd.Series({
        "Vbr": Vbr,
        "Jref": Jref,
        "Npts": len(g)
    })

group_cols = ["HTL","Electrode","StressMode","StressVoltage","StressDuration_s","SourceFile"]
features_df = type1_df.groupby(group_cols).apply(extract_curve_features).reset_index()

C:\Users\Mazhar\AppData\Local\Temp\ipykernel_20028\3041683487.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  features_df = type1_df.groupby(group_cols).apply(extract_curve_features).reset_index()


In [28]:
I_SOFT = 20.0
I_HARD = 50.0
VREF = 1.0

def extract_curve_features(g):
    g = g.sort_values("V")
    V = g["V"].to_numpy()
    J = g["J"].to_numpy()
    absJ = np.abs(J)

    def get_vbr(th):
        idx = np.where(absJ >= th)[0]
        return V[idx[0]] if len(idx) else np.nan

    return pd.Series({
        "Vbr_soft": get_vbr(I_SOFT),
        "Vbr_hard": get_vbr(I_HARD),
        "Jref": J[np.argmin(np.abs(V - VREF))],
        "Npts": len(g)
    })

features_df = type1_df.groupby(group_cols, dropna=False).apply(extract_curve_features).reset_index()


C:\Users\Mazhar\AppData\Local\Temp\ipykernel_20028\4092386711.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  features_df = type1_df.groupby(group_cols, dropna=False).apply(extract_curve_features).reset_index()


In [30]:
from numpy import gradient

VREF = 1.0

def extract_curve_features(g):
    g = g.sort_values("V")
    V = g["V"].to_numpy()
    J = g["J"].to_numpy()
    absJ = np.abs(J)

    # numerical derivative
    dJdV = gradient(absJ, V)

    # find knee: where slope jumps strongly
    slope_norm = dJdV / np.nanmax(dJdV)
    idx = np.where(slope_norm > 0.3)[0]   # 30% of max slope = knee

    Vbr_knee = V[idx[0]] if len(idx) else np.nan
    Jref = J[np.argmin(np.abs(V - VREF))]

    return pd.Series({
        "Vbr_knee": Vbr_knee,
        "Jref": Jref,
        "Npts": len(g)
    })

features_df = type1_df.groupby(group_cols, dropna=False).apply(extract_curve_features).reset_index()
features_df.head()

C:\Users\Mazhar\AppData\Local\Temp\ipykernel_20028\1657116498.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  features_df = type1_df.groupby(group_cols, dropna=False).apply(extract_curve_features).reset_index()


,HTL,Electrode,StressMode,StressVoltage,StressDuration_s,SourceFile,Vbr_knee,Jref,Npts
0,MeO2PACz,Ag,constant,0.8,3600.0,MeO2PACz_Ag_1h_0p8V.csv,1.06,6.720870,71.0
1,MeO2PACz,Ag,constant,0.8,54000.0,MeO2PACz_Ag_15h_0p8V.csv,1.12,14.766414,71.0
2,MeO2PACz,Ag,constant,1.5,54000.0,MeO2PACz_Ag_15h_1p5V.csv,0.06,-29.974597,71.0
3,MeO2PACz,Ag,stepstress,0.2,NaN,MeO2PACz_Ag_stepstress_0p2V.csv,1.14,14.875386,71.0
4,MeO2PACz,Ag,stepstress,0.4,NaN,MeO2PACz_Ag_stepstress_0p4V.csv,1.08,9.975540,71.0


In [32]:
# --- Save outputs ---
timeseries_path = os.path.join(OUT_DIR, "perovai_type1_timeseries_week08.csv")
features_path   = os.path.join(OUT_DIR, "perovai_type1_features_week08.csv")

type1_df.to_csv(timeseries_path, index=False)
features_df.to_csv(features_path, index=False)

print("Saved files:")
print(" -", timeseries_path)
print(" -", features_path)

# quick sanity summary
print("\nSummary:")
print("Timeseries shape:", type1_df.shape)
print("Features shape:", features_df.shape)
print("\nFeature columns:", features_df.columns.tolist())

Saved files:
 - ../results\perovai_type1_timeseries_week08.csv
 - ../results\perovai_type1_features_week08.csv

Summary:
Timeseries shape: (639, 8)
Features shape: (9, 9)

Feature columns: ['HTL', 'Electrode', 'StressMode', 'StressVoltage', 'StressDuration_s', 'SourceFile', 'Vbr_knee', 'Jref', 'Npts']


In [33]:
import os

OUT_DIR = "../results"
os.makedirs(OUT_DIR, exist_ok=True)

# Save time series (full processed data)
type1_df.to_csv(os.path.join(OUT_DIR, "perovai_type1_timeseries_week08.csv"), index=False)

# Save extracted features
features_df.to_csv(os.path.join(OUT_DIR, "perovai_type1_features_week08.csv"), index=False)

print("Saved files:")
print(" - perovai_type1_timeseries_week08.csv")
print(" - perovai_type1_features_week08.csv")

Saved files:
 - perovai_type1_timeseries_week08.csv
 - perovai_type1_features_week08.csv
